In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *
import pandas as pd
from datetime import datetime
import fnmatch


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz')
xd, yd = s['xd'], s['yd']

df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers_cor.csv', skipinitialspace=True).dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()


def make_map(files, dlon=0.2, dsine=1 / 359.5, stonyhurst=False, mu_thr=0.1):
    grid = np.mgrid[-1:1 + dsine / 2:dsine, :360:dlon]
    grid[0] = np.arcsin(grid[0].clip(-1,1)) * 180 / np.pi

    lat = np.arange(-1,1 + dsine / 2, dsine)
    lat = np.arcsin(lat.clip(-1,1)) * 180 / np.pi
    lon = np.arange(0,360,dlon)


    coverage, mean, variance, w, w2 = np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0])

    for file in files[:]:
        print(file)

        with fits.open(file) as hdul:
            header = hdul[0].header.copy()
            data = hdul[0].data.copy()

        did = int(file.split('_')[-1].split('.')[0])
        xd_, yd_ = crop_grid(xd, yd, header)

        view = View.from_header(header)
        view.update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], crota=view.crota + 0.25, rsun=ru_sun[dids == did][0], inplace=True)

        transform = ~ToSpherical() - view.to_synoptic(correct_mu=True, mu_thr=mu_thr, stonyhurst=stonyhurst)

        grid_, alpha = transform(grid)
        grid_ = (interp2d(xd_, *grid_, kind='bilinear'), interp2d(yd_, *grid_, kind='bilinear'))
        data = interp2d(data, *grid_) * alpha

        n = (~np.isnan(data)) * (1 / alpha) ** 4

        coverage += ~np.isnan(data)
        w += np.nan_to_num(n)
        w2 += np.nan_to_num(n ** 2)
        mean_ = mean.copy()
        mean += np.nan_to_num((data - mean) * n / w)
        variance += np.nan_to_num((data - mean) * (data - mean_) * n)

    variance = variance / (1 - w2 / w ** 2) * w2 / w ** 3
    mean[coverage == 0] = np.nan
    variance[coverage == 0] = np.nan
    w[coverage == 0] = np.nan
    coverage[coverage == 0] = np.nan
    return mean, variance, coverage, w, lat, lon

In [3]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/wcs.csv', skipinitialspace=True).dropna()

dates = df['date'].to_numpy()
car_rot = df['car_rot'].to_numpy()
hgln = df['hgln_obs'].to_numpy()

np.unique(car_rot)

array([2279, 2280, 2281, 2282, 2283, 2284, 2285, 2286, 2287, 2288, 2289,
       2290, 2291, 2292, 2293, 2294, 2295, 2296, 2297, 2298, 2299, 2300,
       2301, 2302, 2303, 2304, 2305, 2306])

In [6]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2024_2025/*')))
t = np.where(car_rot == 2296)[0]

mean, variance, coverage, weight, lat, lon = make_map(files[t], dlon=0.5, dsine=1 / 359.5)

/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250408T060003_V202602220258_0544080501.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250408T180003_V202602220258_0544080502.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250409T060003_V202602220258_0544090501.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250409T180003_V202602220258_0544090502.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250410T060002_V202602220258_0544100501.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250410T180003_V202602220258_0544100502.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250411T060003_V202602220258_0544110501.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250411T180003_V202602220258_0544110502.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20250412T060003_V202602220258_0544120501.fits.gz
/home/ulyanov/data/solo/phi/

/tmp/ipykernel_42466/1430487489.py:48: RuntimeWarning: invalid value encountered in divide
  variance = variance / (1 - w2 / w ** 2) * w2 / w ** 3
/tmp/ipykernel_42466/1430487489.py:48: RuntimeWarning: divide by zero encountered in divide
  variance = variance / (1 - w2 / w ** 2) * w2 / w ** 3


In [7]:
plt.figure(figsize=(14,8))
plt.imshow(mean, aspect='auto', origin='lower', cmap='hmimag', vmin=-1000, vmax=1000)
plt.tight_layout()

In [8]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2024_2025/*')))

for i in np.unique(car_rot):
    print(i)

    t = np.where(car_rot == i)[0]
    mean, variance, coverage, weight, lat, lon = make_map(files[t])

    np.savez(f'temp/{i}.npz', mean=mean, variance=variance, coverage=coverage, weight=weight, lat=lat, lon=lon, start=str(dates[t[0]]), end=str(dates[t[-1]]))

2279
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240101T040003_V202602220902_0441010503.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240101T060007_V202602220902_0441010504.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240101T080007_V202602220902_0441010505.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240101T100007_V202602220902_0441010506.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240101T120007_V202602220902_0441010507.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240101T140007_V202602220902_0441010508.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240101T160007_V202602220902_0441010509.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240101T180007_V202602220902_0441010510.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240101T200003_V202602220902_0441010511.fits.gz
/home/ulyanov/data/solo

/tmp/ipykernel_42466/1430487489.py:48: RuntimeWarning: invalid value encountered in divide
  variance = variance / (1 - w2 / w ** 2) * w2 / w ** 3
/tmp/ipykernel_42466/1430487489.py:48: RuntimeWarning: divide by zero encountered in divide
  variance = variance / (1 - w2 / w ** 2) * w2 / w ** 3


2280
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240116T180006_V202602220902_0441160507.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240116T210006_V202602220902_0441160508.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240117T000009_V202602220902_0441170501.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240117T030009_V202602220902_0441170502.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240117T060006_V202602220902_0441170503.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240117T090006_V202602220902_0441170504.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240117T120009_V202602220902_0441170505.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240117T150006_V202602220902_0441170506.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20240118T000009_V202602220902_0441180501.fits.gz
/home/ulyanov/data/solo

In [ ]:
plt.figure(figsize=(14,8))
plt.imshow(weight / coverage, aspect='auto', origin='lower')
plt.tight_layout()

In [ ]:
plt.figure(figsize=(14,8))
plt.imshow(mean, aspect='auto', origin='lower', cmap='hmimag', vmin=-1000, vmax=1000)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(14,8))
plt.imshow(np.sqrt(variance.clip(0)), aspect='auto', origin='lower', vmin=0, vmax=10)
plt.tight_layout()

In [ ]:
files = sorted(glob.glob('temp/*'))

Q = []
for file in files:
    s = np.load(file)
    data = s['data']
    Q += [np.nanmean(data, axis=1)]

Q = np.array(Q)

In [ ]:
plt.figure(figsize=(14,8))
plt.imshow(Q.T, 'bwr', aspect='auto', origin='lower', vmin=-10, vmax=10, interpolation='bicubic')
plt.tight_layout()


In [ ]:
plt.figure(figsize=(10,8))
plt.plot(lat, Q[18])
plt.plot(lat, Q[23])

plt.ylim(-15,15)
plt.tight_layout()